## Menyiapkan Dataset

In [1]:
import numpy as np
import pandas as pd

np.random.seed(212)
n = 3000
menu_list = ["Kopi Susu", "Americano", "Cappuccino", "Roti Bakar", "Kentang Goreng", "Es Teh"]
cabang_list = ["Magelang", "Yogyakarta", "Semarang"]
waktu_list = ["Pagi", "Siang", "Sore", "Malam"]

data_tugas12 = pd.DataFrame({
    "nama_menu": np.random.choice(menu_list, size=n),
    "cabang": np.random.choice(cabang_list, size=n),
    "waktu_pesan": np.random.choice(waktu_list, size=n),
    "qty": np.random.randint(1, 6, size=n),
    "waktu_tunggu_menit": np.random.randint(2, 25, size=n),
})

prob_positif = 1 / (1 + np.exp((data_tugas12["waktu_tunggu_menit"] - 12) / 4))
data_tugas12["ulasan_positif"] = (np.random.rand(n) < prob_positif).astype(int)

data_tugas12.to_csv("tugas12_ulasan.csv", index=False)
print(f"Dataset berhasil dibuat: {len(data_tugas12)} baris")
print("Distribusi label:")
print(data_tugas12["ulasan_positif"].value_counts())

Dataset berhasil dibuat: 3000 baris
Distribusi label:
ulasan_positif
0    1610
1    1390
Name: count, dtype: int64


## Membuat SparkSession

In [2]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col

spark = SparkSession.builder \
    .appName("Tugas12") \
    .master("local[*]") \
    .getOrCreate()
spark.sparkContext.setLogLevel("ERROR")
spark.conf.set("spark.sql.shuffle.partitions", "4")

print("SparkSession siap. Versi Spark:", spark.version)

26/11/25 15:10:14 WARN Utils: Your hostname, indra resolves to a loopback address: 127.0.1.1; using 10.90.68.151 instead (on interface wlo1)
26/11/25 15:10:14 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/11/25 15:10:15 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


SparkSession siap. Versi Spark: 3.5.9


## Membaca Dataset

In [3]:
df = spark.read.csv("tugas12_ulasan.csv", header=True, inferSchema=True)
print("Total baris:", df.count())
df.show(5)
df.groupBy("ulasan_positif").count().show()

Total baris: 3000
+--------------+----------+-----------+---+------------------+--------------+
|     nama_menu|    cabang|waktu_pesan|qty|waktu_tunggu_menit|ulasan_positif|
+--------------+----------+-----------+---+------------------+--------------+
|     Kopi Susu|Yogyakarta|       Sore|  5|                 5|             1|
|        Es Teh|Yogyakarta|       Pagi|  1|                21|             0|
|    Cappuccino|Yogyakarta|       Sore|  5|                15|             0|
|Kentang Goreng|  Semarang|      Malam|  2|                 5|             1|
|     Americano|Yogyakarta|      Malam|  2|                10|             0|
+--------------+----------+-----------+---+------------------+--------------+
only showing top 5 rows

+--------------+-----+
|ulasan_positif|count|
+--------------+-----+
|             1| 1390|
|             0| 1610|
+--------------+-----+



## A. Membangun ML Pipeline

In [4]:
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler
from pyspark.ml.classification import LogisticRegression
from pyspark.ml import Pipeline

# StringIndexer untuk kolom kategorikal
idx_menu = StringIndexer(inputCol="nama_menu", outputCol="menu_idx")
idx_cabang = StringIndexer(inputCol="cabang", outputCol="cabang_idx")
idx_waktu = StringIndexer(inputCol="waktu_pesan", outputCol="waktu_idx")

# OneHotEncoder untuk ketiga kolom indeks
ohe = OneHotEncoder(
    inputCols=["menu_idx", "cabang_idx", "waktu_idx"],
    outputCols=["menu_vec", "cabang_vec", "waktu_vec"]
)

# VectorAssembler seluruh fitur
assembler = VectorAssembler(
    inputCols=["qty", "waktu_tunggu_menit", "menu_vec", "cabang_vec", "waktu_vec"],
    outputCol="features"
)

lr = LogisticRegression(featuresCol="features", labelCol="ulasan_positif")

pipeline = Pipeline(stages=[idx_menu, idx_cabang, idx_waktu, ohe, assembler, lr])
print("Pipeline berhasil dibuat.")

Pipeline berhasil dibuat.


## B. Pelatihan & Evaluasi

In [5]:
from pyspark.ml.evaluation import MulticlassClassificationEvaluator, BinaryClassificationEvaluator

# Split data 80% training dan 20% testing (seed=77)
train, test = df.randomSplit([0.8, 0.2], seed=77)
print(f"Data training: {train.count()} | Data testing: {test.count()}")

model = pipeline.fit(train)
predictions = model.transform(test)

predictions.select("nama_menu", "cabang", "waktu_pesan", "qty", "waktu_tunggu_menit", "ulasan_positif", "prediction", "probability").show(10, truncate=False)

# Evaluasi model
ev_acc = MulticlassClassificationEvaluator(labelCol="ulasan_positif", predictionCol="prediction", metricName="accuracy")
ev_f1 = MulticlassClassificationEvaluator(labelCol="ulasan_positif", predictionCol="prediction", metricName="f1")
ev_auc = BinaryClassificationEvaluator(labelCol="ulasan_positif", metricName="areaUnderROC")

accuracy = ev_acc.evaluate(predictions)
f1 = ev_f1.evaluate(predictions)
auc = ev_auc.evaluate(predictions)

print(f"Accuracy : {accuracy:.4f}")
print(f"F1-score : {f1:.4f}")
print(f"AUC-ROC  : {auc:.4f}")

# Confusion matrix sederhana
predictions.groupBy("ulasan_positif", "prediction").count().orderBy("ulasan_positif", "prediction").show()

Data training: 2411 | Data testing: 589
+---------+--------+-----------+---+------------------+--------------+----------+----------------------------------------+
|nama_menu|cabang  |waktu_pesan|qty|waktu_tunggu_menit|ulasan_positif|prediction|probability                             |
+---------+--------+-----------+---+------------------+--------------+----------+----------------------------------------+
|Americano|Magelang|Malam      |1  |9                 |1             |1.0       |[0.3058092647053278,0.6941907352946721] |
|Americano|Magelang|Malam      |1  |15                |1             |0.0       |[0.6253879175414356,0.37461208245856437]|
|Americano|Magelang|Malam      |2  |2                 |1             |1.0       |[0.08601505227726759,0.9139849477227324]|
|Americano|Magelang|Malam      |3  |4                 |1             |1.0       |[0.12916081752315708,0.8708391824768429]|
|Americano|Magelang|Malam      |3  |13                |0             |0.0       |[0.522484905890237

## C. Model Pembanding

In [7]:
from pyspark.ml.classification import RandomForestClassifier

rf = RandomForestClassifier(featuresCol="features", labelCol="ulasan_positif", numTrees=30)
pipeline_rf = Pipeline(stages=[idx_menu, idx_cabang, idx_waktu, ohe, assembler, rf])

model_rf = pipeline_rf.fit(train)
predictions_rf = model_rf.transform(test)

accuracy_rf = ev_acc.evaluate(predictions_rf)
f1_rf = ev_f1.evaluate(predictions_rf)
auc_rf = ev_auc.evaluate(predictions_rf)

print("=== Perbandingan Hasil ===")
print(f"Logistic Regression -> Accuracy: {accuracy:.4f} | F1: {f1:.4f} | AUC-ROC: {auc:.4f}")
print(f"Random Forest       -> Accuracy: {accuracy_rf:.4f} | F1: {f1_rf:.4f} | AUC-ROC: {auc_rf:.4f}")

=== Perbandingan Hasil ===
Logistic Regression -> Accuracy: 0.7929 | F1: 0.7926 | AUC-ROC: 0.8650
Random Forest       -> Accuracy: 0.7963 | F1: 0.7959 | AUC-ROC: 0.8457


Dari perbandingan ketiga metrik tersebut, Random Forest memiliki nilai accuracy (0.7963 vs 0.7929) dan F1-score (0.7959 vs 0.7926) yang sedikit lebih tinggi, tetapi Logistic Regression lebih unggul pada nilai AUC-ROC (0.8650 vs 0.8457). Jadi secara keseluruhan Logistic Regression lebih baik dalam membedakan kelas positif dan negatif pada data ini.

## D. Analisis Kesalahan Model

In [8]:
# Model terbaik berdasarkan AUC tertinggi adalah Logistic Regression
kesalahan = predictions.filter(col("ulasan_positif") != col("prediction"))
print("Jumlah kesalahan prediksi:", kesalahan.count())

kesalahan.select("nama_menu", "cabang", "waktu_pesan", "qty", "waktu_tunggu_menit", "ulasan_positif", "prediction", "probability").show(10, truncate=False)
kesalahan.describe("waktu_tunggu_menit").show()

Jumlah kesalahan prediksi: 122
+---------+--------+-----------+---+------------------+--------------+----------+----------------------------------------+
|nama_menu|cabang  |waktu_pesan|qty|waktu_tunggu_menit|ulasan_positif|prediction|probability                             |
+---------+--------+-----------+---+------------------+--------------+----------+----------------------------------------+
|Americano|Magelang|Malam      |1  |15                |1             |0.0       |[0.6253879175414356,0.37461208245856437]|
|Americano|Magelang|Siang      |2  |22                |1             |0.0       |[0.9113822566149019,0.08861774338509809]|
|Americano|Magelang|Sore       |1  |7                 |0             |1.0       |[0.23209137939008503,0.7679086206099149]|
|Americano|Magelang|Sore       |2  |11                |0             |1.0       |[0.42615454016906074,0.5738454598309393]|
|Americano|Magelang|Sore       |3  |5                 |0             |1.0       |[0.1653400232331677,0.83465

Dari 10 baris kesalahan dan ringkasan statistik di atas, terlihat bahwa kesalahan prediksi banyak terjadi di sekitar `waktu_tunggu_menit` bernilai 10 sampai 15 menit (rata-ratanya sekitar 12,65 menit). Pada rentang tersebut nilai probabilitas model masih berada di sekitar 0,5 sehingga prediksi lebih mudah meleset ketika ada variasi preferensi pelanggan. Sedangkan untuk waktu tunggu yang sangat singkat atau sangat lama, kesalahan tetap bisa muncul sebagai kasus pengecualian saja.

## E. Kesimpulan Bisnis

Berdasarkan model yang sudah dibuat, faktor yang paling berpengaruh terhadap peluang pelanggan memberikan ulasan positif di Kopi Nusantara adalah lama waktu tunggu pesanan (waktu_tunggu_menit), sedangkan pilihan menu, cabang, waktu pesan, dan jumlah pesanan pengaruhnya jauh lebih kecil. Seperti yang terlihat pada bagian D, area paling rawan membuat kepuasan pelanggan berubah dari positif menjadi negatif berada di sekitar ambang batas 10 hingga 15 menit dengan rata-rata kesalahan di angka 12,65 menit. Untuk meningkatkan kepuasan pelanggan, manajemen Kopi Nusantara sebaiknya menetapkan batas waktu penyajian maksimal sekitar 10 sampai 12 menit di seluruh cabang. Selain itu, alur kerja barista dan dapur pada jam ramai bisa dioptimalkan dengan memisahkan antrean minuman yang cepat dibuat dari menu makanan seperti Roti Bakar atau Kentang Goreng yang butuh waktu lebih lama. Jika antrean sedang padat dan waktu tunggu diperkirakan melewati 12 menit, staf kasir juga dapat langsung menginformasikan estimasi waktu kepada pelanggan sejak awal atau memberikan kompensasi ringan agar pelanggan tidak merasa kecewa.

## Penutup

In [9]:
spark.stop()
print("SparkSession ditutup.")

SparkSession ditutup.
